# Weekly stock-price forecasting: ARIMA, neural network, and naive baseline

A cleaned coursework study of 2023 weekly adjusted closing prices for ticker `005380.KS`, with a four-week January 2024 one-step forecast evaluation. This is an educational model comparison, **not investment advice**. Data are fetched via yfinance or loaded from a local CSV cache.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor

TICKER = "005380.KS"
CACHE = Path("data/005380_KS_2023_2024.csv")
if CACHE.exists():
    daily = pd.read_csv(CACHE, index_col=0, parse_dates=True)
else:
    daily = yf.download(TICKER, start="2023-01-01", end="2024-02-01",
                        auto_adjust=False, multi_level_index=False, progress=False)
    if daily.empty:
        raise RuntimeError("Download returned no data. Supply the cached CSV described in README.")
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    daily.to_csv(CACHE)
if "Adj Close" not in daily.columns:
    raise ValueError("Expected an 'Adj Close' column in the daily price data.")
weekly = daily["Adj Close"].dropna().resample("W-FRI").mean().dropna()
train = weekly.loc["2023-01-01":"2023-12-31"]
test = weekly.loc["2024-01-01":"2024-01-31"]
if len(train) < 30 or len(test) < 2:
    raise ValueError(f"Need enough 2023/January 2024 weeks; got {len(train)} and {len(test)}.")
print(f"Train: {len(train)} weeks | Test: {len(test)} weeks")


## Explore stationarity

The ADF test is a diagnostic, not proof of forecast quality. ARIMA order is kept at (1,1,1) to reflect the walk-forward portion of the original notebook.


In [ ]:
print("ADF p-value (levels):", adfuller(train)[1])
print("ADF p-value (first difference):", adfuller(train.diff().dropna())[1])
ax = train.plot(figsize=(10, 3), label="2023 train")
test.plot(ax=ax, label="January 2024 test")
ax.set(xlabel="Week ending", ylabel="Weekly mean adjusted close (KRW)")
ax.legend()
plt.tight_layout()
plt.show()


## One-step rolling evaluation

For each January week, ARIMA refits on all observations available before that week. The naive baseline predicts the preceding observed week. The neural network is fitted on 2023 only, using two lagged weekly prices; its January predictions use observed preceding weeks, without retraining. These are one-step predictions, not a four-week forecast made in December.


In [ ]:
lagged_train = pd.DataFrame({"lag1":train.shift(1), "lag2":train.shift(2), "target":train}).dropna()
mlp = TransformedTargetRegressor(
    regressor=make_pipeline(StandardScaler(), MLPRegressor(
        hidden_layer_sizes=(12,), random_state=42, max_iter=2000)),
    transformer=StandardScaler())
mlp.fit(lagged_train[["lag1", "lag2"]], lagged_train["target"])

history = train.copy()
rows = []
for date, actual in test.items():
    arima_pred = float(ARIMA(history, order=(1, 1, 1)).fit().forecast(steps=1).iloc[0])
    naive_pred = float(history.iloc[-1])
    features = pd.DataFrame({"lag1":[history.iloc[-1]], "lag2":[history.iloc[-2]]})
    mlp_pred = float(mlp.predict(features)[0])
    rows.append({"week":date, "actual":float(actual), "ARIMA(1,1,1)":arima_pred,
                 "Naive":naive_pred, "Neural network":mlp_pred})
    history = pd.concat([history, pd.Series([actual], index=[date])])
results = pd.DataFrame(rows).set_index("week")
print(results.round(2))


In [ ]:
metrics = []
for name in ["ARIMA(1,1,1)", "Naive", "Neural network"]:
    metrics.append({"model":name,
                    "MAE":mean_absolute_error(results["actual"], results[name]),
                    "RMSE":np.sqrt(mean_squared_error(results["actual"], results[name]))})
print(pd.DataFrame(metrics).set_index("model").round(2))
ax = results.plot(marker="o", figsize=(10, 4), title="January 2024: one-step weekly predictions")
ax.set(xlabel="Week ending", ylabel="Weekly mean adjusted close (KRW)")
plt.tight_layout()
plt.show()


## Interpretation

The original notebook reported ARIMA walk-forward RMSE of 5,089.875 for four January weeks and neural-network test R² of 0.380 for a different 2023 holdout. These metrics come from **different evaluation windows** and should not be compared directly. The cleaned notebook recomputes all three approaches on the same January weeks; results may differ because of download revisions, weekly aggregation, and changed neural-network preprocessing. Four test weeks are too few to establish a robust forecasting ranking.
